In [ ]:
# !pip install -U langchain
# !pip install -U langchain-cohere
# !pip install -U langchain-community
# !pip install -U langchain-text-splitters
# !pip install -U langchain-chroma
# !pip install -U pypdf
# !pip install -U chromadb
# !pip install -U gradio

In [ ]:
from langchain_community.document_loaders import PyPDFLoader
from google.colab import userdata
from google.colab import files
from langchain_cohere import ChatCohere
from langchain_cohere import CohereEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.prompts import ChatPromptTemplate

/tmp/ipykernel_469/1208857428.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


In [ ]:
key = userdata.get('coherekey')

In [ ]:
uploaded = files.upload()

pdf_paths = [name for name in uploaded.keys() if name.lower().endswith(".pdf")]
print("Uploaded PDFs:")
for p in pdf_paths:
    print(" -", p)

Saving Arwa_Ahmed_Mohamed_CV.pdf to Arwa_Ahmed_Mohamed_CV.pdf
Uploaded PDFs:
 - Arwa_Ahmed_Mohamed_CV.pdf


In [ ]:
documents = []

for path in pdf_paths:
    loader = PyPDFLoader(path)
    docs = loader.load()
    for d in docs:
        d.metadata["source_file"] = path
    documents.extend(docs)
    print(f"{path}: {len(docs)} pages loaded")

print("\nTotal pages loaded across all PDFs:", len(documents))

Arwa_Ahmed_Mohamed_CV.pdf: 2 pages loaded

Total pages loaded across all PDFs: 2


In [ ]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=100
)

chunks = text_splitter.split_documents(documents)
print("Total chunks:", len(chunks))

Total chunks: 7


In [ ]:
embeddings = CohereEmbeddings(
    model="embed-v4.0",
    cohere_api_key=key
)

In [ ]:
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    collection_name="multi_pdf_rag"
)

In [ ]:
retriever = vectorstore.as_retriever(
    search_kwargs={"k": 5}
)

In [ ]:
llm = ChatCohere(
    model="command-a-03-2025",
    temperature=0,
    cohere_api_key=key
)

In [ ]:
prompt = ChatPromptTemplate.from_template("""
You are a helpful assistant answering questions about a set of uploaded PDF documents.

Answer the user's question using ONLY the context below.
If the answer cannot be found in the context, say "I don't know." Do not invent information.

Context:
{context}

Question:
{question}
""")


In [ ]:
def format_docs_with_sources(docs):
    formatted = []
    for doc in docs:
        source = doc.metadata.get("source_file", "unknown file")
        page = doc.metadata.get("page", None)
        page_label = f"page {page + 1}" if isinstance(page, int) else "page unknown"
        formatted.append(f"[{source} — {page_label}]\n{doc.page_content}")
    return "\n\n".join(formatted)


def get_sources(docs):
    seen = []
    for doc in docs:
        source = doc.metadata.get("source_file", "unknown file")
        page = doc.metadata.get("page", None)
        page_label = f"p.{page + 1}" if isinstance(page, int) else "p.?"
        label = f"{source} ({page_label})"
        if label not in seen:
            seen.append(label)
    return seen


def ask(question: str, active_retriever=None):
    active_retriever = active_retriever or retriever
    docs = active_retriever.invoke(question)
    context = format_docs_with_sources(docs)
    messages = prompt.format_messages(context=context, question=question)
    response = llm.invoke(messages)
    sources = get_sources(docs)
    return response.content, sources

In [ ]:
question1 = "What is this set of documents about?"
answer, sources = ask(question1)

print("Q:", question1)
print("A:", answer)

Q: What is this set of documents about?
A: This set of documents is about **Arwa Ahmed Mohamed's CV**, which details her **skills**, **experience**, **projects**, and **internships**. It highlights her expertise in areas such as **Natural Language Processing (NLP)**, **software development**, **UI/UX design**, and **programming languages** like **C++**, **Java**, **Python**, and **JavaScript**. The CV also includes descriptions of her projects, such as a **healthcare platform**, **messaging app**, **food ordering system**, and **RAG system**, as well as her internships in **NLP**, **AI**, **frontend development**, and **UI/UX design**.


In [ ]:
question2 = "list skills"
answer, sources = ask(question2)

print("Q:", question2)
print("A:", answer)

Q: list skills
A: Based on the context provided, here is a list of Arwa Ahmed Mohamed's skills:

**Programming Languages:**

* C++
* Java
* C#
* Python
* JavaScript
* TypeScript

**Technical Basics:**

* Object-Oriented Programming (OOP)
* Data Structures
* Algorithms
* Database Design (SQL/Oracle)

**Tools:**

* Git
* GitHub

**Frontend:**

* HTML5
* CSS3
* React (mentioned in experience section)

**Design:**

* Figma (Design Systems, Prototyping, Auto Layout)

**Soft Skills:**

* Rapid Learning
* Problem Solving
* Team Collaboration
* Task Prioritization
* Creative Thinking

**Additional Skills (from experience):**

* Natural Language Processing (NLP)
* Deep Learning Architectures (NNs, RNNs)
* Transformers
* Large Language Models (LLMs)
* Building RAG pipelines
* Pydantic AI
* LlamaIndex (RAG)
* LangGraph
* Streamlit
* FastAPI
* n8n workflow automation
* UI/UX Design (wireframing, prototyping, scalable components, Auto Layout, Color Theory, Typography)
* Software Quality Assurance a

In [ ]:
question3 = "list the skills mentioned in the file that work with AI Engineering Track"
answer, sources = ask(question3)

print("Q:", question3)
print("A:", answer)

Q: list the skills mentioned in the file that work with AI Engineering Track
A: Based on the context provided, the skills mentioned in the file that are relevant to AI Engineering include:

1. **Natural Language Processing (NLP)**  
2. **Data Structures & Algorithms**  
3. **Machine Learning (ML)**  
4. **Deep Learning Architectures (NNs, RNNs, Transformers)**  
5. **Large Language Models (LLMs)**  
6. **Retrieval-Augmented Generation (RAG) Pipelines**  
7. **Vectorization and Embeddings**  
8. **Text Preprocessing**  
9. **Python**  
10. **Pydantic AI**  
11. **LlamaIndex (RAG)**  
12. **LangGraph**  
13. **FastAPI**  
14. **FAISS Vector Database**  
15. **OpenAI LLM**  

These skills are directly related to AI Engineering and are mentioned in the context of Arwa Ahmed Mohamed's CV.


In [ ]:
question4 = "tell me if there's any certificates about AI"
answer, sources = ask(question4)

print("Q:", question4)
print("A:", answer)

Q: tell me if there's any certificates about AI
A: Based on the provided context, there are no certificates specifically mentioned related to AI. The certificates listed are:

1. React Frontend Development from Digital Egypt Pioneers Initiative (DEPI)
2. Business English Track from Digital Egypt Pioneers Initiative (DEPI)
3. IT (Quality Testing) from Telecom Egypt (WE)
4. Blender 3D Workshop from OSC - Open Source Community
5. Canva Essentials from Canva Team

None of these certificates are explicitly related to AI.


In [ ]:
question5 = "what is SDLC?"
answer, sources = ask(question5)

print("Q:", question3)
print("A:", answer)

Q: list the skills mentioned in the file that work with AI Engineering Track
A: I don't know. The context provided does not contain any information about SDLC (Software Development Life Cycle).
